In [ ]:
# =======================================================
# CELL 1 - GPU CHECK + DRIVE MOUNT
# Run first. If GPU is missing, stop here and change runtime.
# =======================================================
import os, sys, subprocess
from pathlib import Path
import torch
from google.colab import drive

drive.mount('/content/drive', force_remount=False)

if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"OK GPU: {name} ({vram:.1f} GB VRAM)")
else:
    raise RuntimeError(
        "\nNo GPU detected.\n"
        "Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU\n"
        "Then re-run this cell."
    )

print("OK Drive mounted.")

In [ ]:
# =======================================================
# CELL 2 - INSTALL ALL DEPENDENCIES
# Safe to re-run. Takes a few minutes on first run.
# =======================================================
import sys, subprocess

packages = [
    "torch", "numpy", "PyYAML", "pytest", "tqdm",
    "sympy", "scipy", "psutil", "networkx",
    "fastapi", "uvicorn", "httpx",
    "sentence-transformers",
    "transformers", "accelerate", "tokenizers",
]

print("Installing standard packages...")
for pkg in packages:
    r = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", pkg],
        capture_output=True, text=True
    )
    icon = "OK" if r.returncode == 0 else "WARN"
    print(f"  {icon} {pkg}")

print("
Installing Muon optimizer from GitHub...")
r = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/KellerJordan/Muon"],
    capture_output=True, text=True
)
if r.returncode == 0:
    print("  OK muon")
else:
    print("  WARN muon install failed - AdamW fallback will be used automatically")

print("
OK All dependencies installed.")


In [ ]:
# =======================================================
# CELL 3 - CLONE OR UPDATE AN-RA
# Clones fresh if not present. Pulls latest if already there.
# =======================================================
import os, sys, subprocess
from pathlib import Path

REPO = Path("/content/An-Ra")
GITHUB_URL = "https://github.com/dhurv0045com-spec/An-Ra-the-new-AGI.git"

if not REPO.exists():
    print("Cloning An-Ra (first time)...")
    r = subprocess.run(["git", "clone", GITHUB_URL, str(REPO)], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"Clone failed:\n{r.stderr}")
    print("OK Cloned.")
else:
    r = subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], capture_output=True, text=True)
    print(f"OK {r.stdout.strip() or 'Already up to date.'}")

os.environ["ANRA_REPO_ROOT"] = str(REPO)
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from anra_paths import inject_all_paths, ensure_dirs
inject_all_paths()
ensure_dirs()
print(f"OK Paths injected. Working dir: {REPO}")

In [ ]:
# =======================================================
# CELL 4 - RESTORE V2 ARTIFACTS
# Uses repo files first, restores from Drive if available.
# =======================================================
import os, sys, shutil, json
from pathlib import Path

REPO = Path("/content/An-Ra")
DRIVE = Path("/content/drive/MyDrive/AnRa")
V2_DRIVE = DRIVE / "v2"

if not REPO.exists():
    raise RuntimeError(
        f"Repo not found at {REPO}.
"
        "Run the clone/update cell first."
    )

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

os.chdir(REPO)

from anra_paths import inject_all_paths, ensure_dirs, TOKENIZER_DIR, TRAINING_DATA_DIR
inject_all_paths()
ensure_dirs()

print("=== Restoring An-Ra V2 artifacts from Drive ===
")

for name in ["anra_v2_brain.pt", "anra_v2_identity.pt", "anra_v2_ouroboros.pt"]:
    local_path = REPO / name
    drive_path = V2_DRIVE / "checkpoints" / name
    if local_path.exists() and local_path.stat().st_size > 10_000:
        print(f"  OK {name} already present in repo ({local_path.stat().st_size / 1e6:.1f} MB)")
    elif drive_path.exists() and drive_path.stat().st_size > 10_000:
        shutil.copy2(drive_path, local_path)
        print(f"  OK {name} restored from Drive ({local_path.stat().st_size / 1e6:.1f} MB) -> will resume training")
    else:
        print(f"  INFO {name}: not found locally or on Drive -> fresh start")

TOKENIZER_DIR.mkdir(parents=True, exist_ok=True)
tok_local = TOKENIZER_DIR / "tokenizer_v2.json"
tok_meta_local = TOKENIZER_DIR / "tokenizer_v2.json.meta.json"
tok_drive = V2_DRIVE / "tokenizer_v2.json"
tok_meta_drive = V2_DRIVE / "tokenizer_v2.json.meta.json"

if tok_local.exists() and tok_local.stat().st_size > 100:
    print(f"
  OK tokenizer_v2.json already present in repo")
elif tok_drive.exists() and tok_drive.stat().st_size > 100:
    shutil.copy2(tok_drive, tok_local)
    if tok_meta_drive.exists():
        shutil.copy2(tok_meta_drive, tok_meta_local)
    print(f"
  OK tokenizer_v2.json restored from Drive")
else:
    print("
  INFO tokenizer_v2.json not found locally or on Drive -> build_brain.py will auto-create it")

ds_local = TRAINING_DATA_DIR / "anra_dataset_v6_1.txt"
ds_drive = DRIVE / "anra_dataset_v6_1.txt"

if ds_local.exists() and ds_local.stat().st_size > 100_000:
    print(f"
  OK Dataset already present in repo: {ds_local} ({ds_local.stat().st_size / 1e6:.2f} MB)")
elif ds_drive.exists() and ds_drive.stat().st_size > 100_000:
    TRAINING_DATA_DIR.mkdir(parents=True, exist_ok=True)
    shutil.copy2(ds_drive, ds_local)
    print(f"
  OK Dataset restored from Drive: {ds_local} ({ds_local.stat().st_size / 1e6:.2f} MB)")
else:
    print(
        f"
  WARN Dataset not found locally or on Drive.
"
        f"    Expected local: {ds_local}
"
        f"    Expected drive: {ds_drive}
"
        f"    Training cannot start until the dataset exists."
    )

print("
OK V2 artifact restoration complete.")


In [ ]:
# =======================================================
# CELL 5 - V2 SYSTEM HEALTH CHECK
# Verify the new V2 mainline before training starts.
# =======================================================
import os, sys, pickle, subprocess, torch
from pathlib import Path
from google.colab import drive

REPO = Path("/content/An-Ra")
GITHUB_URL = "https://github.com/dhurv0045com-spec/An-Ra-the-new-AGI.git"
if not Path("/content/drive/MyDrive").exists():
    drive.mount('/content/drive', force_remount=False)
if not REPO.exists():
    subprocess.run(["git", "clone", GITHUB_URL, str(REPO)], check=True)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
os.chdir(REPO)

from anra_paths import inject_all_paths, ensure_dirs, TRAINING_DATA_DIR, TOKENIZER_DIR
inject_all_paths()
ensure_dirs()

print("================ AN-RA V2 HEALTH CHECK ================
")

critical = {
    "anra_brain.py": REPO / "anra_brain.py",
    "build_brain.py": REPO / "scripts" / "build_brain.py",
    "train_unified.py": REPO / "training" / "train_unified.py",
    "subword_tokenizer.py": REPO / "tokenizer" / "subword_tokenizer.py",
    "Training dataset": TRAINING_DATA_DIR / "anra_dataset_v6_1.txt",
}
print("-- Critical Files --")
for label, path in critical.items():
    if path.exists():
        size = f" ({path.stat().st_size / 1e6:.2f} MB)" if path.suffix in {".py", ".txt"} else ""
        print(f"  OK {label}{size}")
    else:
        print(f"  FAIL {label}: MISSING - {path}")

print("
-- V2 Tokenizer --")
tok_path = TOKENIZER_DIR / "tokenizer_v2.json"
if tok_path.exists():
    print(f"  OK tokenizer_v2.json present ({tok_path.stat().st_size / 1e6:.2f} MB)")
else:
    print("  INFO tokenizer_v2.json: will be auto-built at training start")

print("
-- V2 Checkpoints --")
for name in ["anra_v2_brain.pt", "anra_v2_identity.pt", "anra_v2_ouroboros.pt"]:
    ckpt = REPO / name
    if ckpt.exists():
        print(f"  OK {name} ({ckpt.stat().st_size / 1e6:.1f} MB)")
    else:
        print(f"  INFO {name}: not found yet")

print("
-- Trainer Status --")
r = subprocess.run([sys.executable, "-m", "training.train_unified", "--mode", "status"], capture_output=True, text=True)
print(r.stdout.strip())
if r.returncode != 0:
    print(r.stderr.strip())

print("
-- GPU --")
if torch.cuda.is_available():
    used = torch.cuda.memory_allocated() / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"  OK {torch.cuda.get_device_name(0)} {used:.1f}/{total:.1f} GB")
else:
    print("  FAIL GPU not available - go to Runtime -> Change runtime type -> T4 GPU")

print("
OK V2 health check complete. If dataset is present, proceed to Cell 6.")


In [ ]:
# =======================================================
# CELL 6 - TRAIN AN-RA V2 (DAILY SESSION)
# 30-minute base session. Auto-resumes from V2 checkpoint.
# =======================================================
import os, sys, subprocess, json
from pathlib import Path
from google.colab import drive

REPO = Path("/content/An-Ra")
GITHUB_URL = "https://github.com/dhurv0045com-spec/An-Ra-the-new-AGI.git"
if not Path("/content/drive/MyDrive").exists():
    drive.mount('/content/drive', force_remount=False)
if not REPO.exists():
    subprocess.run(["git", "clone", GITHUB_URL, str(REPO)], check=True)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
os.chdir(REPO)

from anra_paths import inject_all_paths, TRAINING_DATA_DIR
inject_all_paths()

dataset_path = TRAINING_DATA_DIR / "anra_dataset_v6_1.txt"
if not dataset_path.exists() or dataset_path.stat().st_size < 100_000:
    raise RuntimeError(
        f"Training dataset missing or too small: {dataset_path}
"
        "Run Cell 4 first and make sure anra_dataset_v6_1.txt exists in the repo or on Drive."
    )

print("========================================================")
print("AN-RA V2 DAILY TRAINING SESSION")
print("30 min | auto-resume | own-data-first | saves to Drive")
print("========================================================
")

cmd = [
    sys.executable, "-m", "training.train_unified",
    "--mode", "session",
    "--session_minutes", "30",
    "--checkpoint_path", "anra_v2_brain.pt",
    "--block_size", "256",
    "--batch_size", "32",
]

print("Command:", " ".join(cmd))
print("-" * 58)

proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    cwd=str(REPO),
    bufsize=1,
)
assert proc.stdout is not None
for line in proc.stdout:
    print(line, end="", flush=True)
proc.wait()

print("-" * 58)

if proc.returncode == 0:
    print("
OK V2 daily session complete. Checkpoint saved to Drive.")
else:
    raise RuntimeError(f"V2 training exited with code {proc.returncode}. Read the log above.")

for name in [
    "v2_unified_training_report.json",
    "v2_session_train_metrics.json",
    "v2_next_session_curriculum.json",
    "v2_eval_summary.json",
]:
    rp = REPO / "output" / "v2" / name
    if rp.exists():
        print(f"OK report: {rp.name}")

drive_ckpt = Path("/content/drive/MyDrive/AnRa/v2/checkpoints/anra_v2_brain.pt")
if drive_ckpt.exists():
    print(f"
OK Drive backup: {drive_ckpt.stat().st_size / 1e6:.1f} MB")
    print("Come back anytime - Cell 6 will resume from this V2 checkpoint.")
else:
    print(f"
WARN Drive backup not found at expected path: {drive_ckpt}")

print("
For a milestone run later, use:")
print("python -m training.train_unified --mode train --session_minutes 30")


In [ ]:
# =======================================================
# CELL 7 - POST-TRAINING V2 VERIFICATION
# Run after Cell 6 or a milestone session.
# =======================================================
import os, sys, subprocess
from pathlib import Path

REPO = Path("/content/An-Ra")
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
    os.chdir(REPO)
    from anra_paths import inject_all_paths
    inject_all_paths()

print("=== V2 Post-Training Verification ===
")

print("-- V2 Status --")
r = subprocess.run([sys.executable, "-m", "training.train_unified", "--mode", "status"], capture_output=True, text=True, cwd=str(REPO))
print(r.stdout.strip())

print("
-- V2 Tests --")
r = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/test_v2_stack.py", "-q", "--tb=short", "--no-header"],
    capture_output=True, text=True, cwd=str(REPO)
)
out = (r.stdout + r.stderr).strip()
print(out[-3000:] if len(out) > 3000 else out)

print("
-- V2 Drive Backup --")
drive = Path("/content/drive/MyDrive/AnRa/v2")
for name in [
    "checkpoints/anra_v2_brain.pt",
    "checkpoints/anra_v2_identity.pt",
    "checkpoints/anra_v2_ouroboros.pt",
    "tokenizer_v2.json",
]:
    p = drive / name
    if p.exists():
        print(f"  OK {name} ({p.stat().st_size / 1e6:.1f} MB)")
    else:
        print(f"  INFO {name}: not on Drive yet")

print("
-- V2 Reports --")
for name in [
    "v2_session_train_metrics.json",
    "v2_hard_examples.json",
    "v2_eval_summary.json",
    "v2_next_session_curriculum.json",
    "v2_unified_training_report.json",
]:
    p = REPO / "output" / "v2" / name
    if p.exists():
        print(f"  OK {name}")
    else:
        print(f"  INFO {name}: missing")

print("
OK Done.")
